# LAB-2681 · Building Quantum-Capable AI Models

**IBM TechXchange 2026 · IBM Research**

General-purpose LLMs write quantum code that *looks* right but often isn't: they reach for APIs
that were removed years ago (`execute`, `qiskit.Aer`, `BasicAer`, `IBMQ`), or ignore what real
hardware can do. Prompting harder doesn't fix that. **Training on verified quantum code does.**

In this notebook you walk through the pipeline we use to make a model quantum-capable:

| # | Stage | What you do here |
|---|---|---|
| 1 | **Synthetic prompts** | Look at Qiskit coding tasks that each ship with *unit tests* |
| 2 | **Mellea: Instruct → Validate → Repair** | Let a model attempt each task, test it, and feed failures back |
| 3 | **Three kinds of training data** | Turn the loop's output into **QA**, **correction** and **fill-in-the-blank** data |
| 4 | **Reinforcement learning** | See how the unit tests become the reward for GRPO training |
| 5 | **Validation** | Compare the base model with the RL-tuned model on your own data |

> **The one idea to keep in mind:** every prompt comes with unit tests. So at every stage we
> don't have to *guess* whether code is correct: we run it and find out.

## 0 · Setup

The lab compares two models:

* **`gemma-4-31b-base`**: Google's Gemma-4-31B-it, as downloaded.
* **`gemma-31b-qiskit-tuned`**: the same model after RL training on Qiskit tasks with the pipeline in this notebook.

A 31-billion-parameter model needs a data-centre GPU, so in this lab room **the models are replaced by
recordings** of their real answers. Your instructor runs the same notebook live against the real models.

> **What's recorded and what's real?** Only *what the model said* is recorded. Everything else runs live on
> your machine: Mellea's Instruct → Validate → Repair loop, running the unit tests, deciding pass or fail,
> building the repair prompt from the test failure, and all of the data generation in section 3.
> A small local server (`replay_server.py`) stands in for the model and answers Mellea's requests with the
> recorded text, through the same **OpenAI-compatible** interface a real hosted model uses.

In [ ]:
### Test your kernel setup and environment

print("Hi techXchange!!")

#### Setup environment variables and local replay server

In [ ]:
import os, json, random
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

import lab_helpers as H

# --- Endpoints: only the instructor's live demo sets these ---
BASE_URL  = os.environ.get("LAB_BASE_URL",  "http://localhost:8000/v1")   # base model
TUNED_URL = os.environ.get("LAB_TUNED_URL", "http://localhost:8001/v1")   # RL-tuned model
BASE_MODEL, TUNED_MODEL = "gemma-4-31b-base", "gemma-31b-qiskit-tuned"
CACHE = Path("data/recorded")

def endpoint_up(url):
    try:
        import urllib.request
        with urllib.request.urlopen(url.rstrip("/") + "/models", timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

LIVE = endpoint_up(BASE_URL) and endpoint_up(TUNED_URL)
USE_CACHE = not LIVE   # replay recorded answers unless the real models are reachable
if not LIVE:
    import replay_server
    REPLAY_URL, _replay = replay_server.start()
    print(f"Replay mode: recorded answers for {len(_replay.answers)} prompts are served at {REPLAY_URL}")
    print("Mellea, the unit tests and the data generation all run for real on this machine.")
else:
    print("Live mode: talking to the real models.")

random.seed(0)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

## 1 · Synthetic prompts with built-in verifiers

Our data starts from **parametrized YAML templates**. A template describes a family of Qiskit
tasks ("build a random circuit with *n* qubits and depth *d*"); filling in the parameters gives many
concrete prompts. Each one comes with:

* **`prompt`**: the coding task, in natural language
* **`imports`**: the imports the test environment provides
* **`unittest`**: automated checks that decide whether an answer is correct
* **`template`**: which template family it came from

Let's load the 20 sample prompts for this lab.

In [ ]:
problems = H.load_jsonl("sample_training_data_20_input.jsonl")
overview = H.describe_prompts(problems)
print(f"{len(problems)} prompts from {overview['template'].nunique()} templates across {overview['topic'].nunique()} topics")
overview

In [ ]:
# --- Visualize the prompt data ---
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
overview["topic"].value_counts().sort_values().plot.barh(ax=axes[0], color="#4C78A8")
axes[0].set_title("Topics covered"); axes[0].set_xlabel("prompts")
axes[1].scatter(overview["prompt_chars"], overview["unit_tests"], color="#4C78A8")
axes[1].set_xscale("log"); axes[1].set_xlabel("prompt length (characters, log scale)"); axes[1].set_ylabel("unit tests")
axes[1].set_title("From one-liners to multi-step tasks")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### Look at one prompt and its tests

Change `i` to browse. Notice how specific the tests are: they check names, qubit counts,
depths, even numerical results. A model can't pass by writing something that only *looks* plausible.

In [ ]:
i = 8   # try 0, 5, 8, 15, 18 ...
p = problems[i]
print("TEMPLATE:", p["template"]); print()
print("PROMPT:\n" + p["prompt"])
H.show_code(p["imports"] or "# (no imports provided)", "Imports available to the solution")
H.show_code(p["unittest"], "Hidden unit tests")

### Try it yourself: the tests are the judge

Write an answer to prompt #6 (*"Make a circuit called qc."*) and run the tests on it.
Then break it on purpose (wrong name, wrong type...) and see what the tests say.

In [ ]:
my_answer = '''
from qiskit import QuantumCircuit
qc = QuantumCircuit(2)
'''
passed, report = H.run_tests(my_answer, problems[6])
print("PASSED ✅" if passed else "FAILED ❌")
if report:
    H.show_text(report[-1200:], "pytest output")

### Why "looks right" isn't enough: deprecated APIs

Here is code that would have worked in Qiskit 0.x and is still everywhere on the internet,
so models trained on web data love to write it. The pipeline runs
[`flake8-qiskit-migration`](https://github.com/qiskit-community/flake8-qiskit-migration) on every
answer to flag exactly this.

In [ ]:
from data_augmentation.core.validation import find_deprecated_apis

legacy = '''
from qiskit import QuantumCircuit, execute, Aer
qc = QuantumCircuit(2); qc.h(0); qc.cx(0, 1); qc.measure_all()
counts = execute(qc, Aer.get_backend("qasm_simulator"), shots=1000).result().get_counts()
'''
modern = '''
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
qc = QuantumCircuit(2); qc.h(0); qc.cx(0, 1); qc.measure_all()
counts = StatevectorSampler().run([qc], shots=1000).result()[0].data.meas.get_counts()
'''
for name, src in [("legacy", legacy), ("modern", modern)]:
    flagged, why = find_deprecated_apis(src)
    print(f"{name:7s} deprecated APIs: {flagged}")
    if why:
        print("   " + why.replace("\n", "\n   "))

## 2 · Mellea: Instruct → Validate → Repair (IVR)

[Mellea](https://github.com/generative-computing/mellea) is IBM's open-source library for
*structured* generative programming: instead of one hand-written prompt, you declare what the
output must satisfy and let Mellea drive the model until it does.

```
            ┌──────────────┐     ┌─────────────────────┐   pass   ┌──────┐
 prompt ──▶ │   INSTRUCT   │ ──▶ │ VALIDATE: run tests │ ───────▶ │ done │
            │  (the model) │     └─────────────────────┘          └──────┘
            └──────────────┘                │ fail
                   ▲                        │
                   └──── REPAIR: task + test failure, try again (up to N times)
```

The code that does this in our pipeline is only a few lines:

```python
session.instruct(
    task_prompt,
    requirements=[req("Code must pass all pytest tests", validation_fn=simple_validate(run_unit_tests))],
    strategy=RepairTemplateStrategy(loop_budget=5),    # 1 try + up to 4 repairs
    return_sampling_results=True,                      # keep EVERY attempt, not just the last
)
```

That last line matters: we keep the **failed** attempts too, because they become training data (for generating correction data).

### Connect Mellea to the model

We use the **base** model here on purpose: a model that already solves everything produces no
failures, and failures are what correction data is made from.

In [ ]:
from data_augmentation.models import GenConfig, DEFAULT_SYSTEM_PROMPT
from data_augmentation.core.llm_client import make_session_factory

cfg = GenConfig(
    model=f"hosted_vllm/{BASE_MODEL}",   # litellm id: provider prefix + served model name
    loop_budget=5,                       # max attempts per prompt
    temperature=0.6,
    max_tokens=4096,
)
# In replay mode Mellea talks to the local replay server instead of the real model; nothing else changes.
MELLEA_URL = BASE_URL if LIVE else REPLAY_URL
session_factory = make_session_factory(MELLEA_URL, os.environ.get("LITELLM_API_KEY", "EMPTY"), None, cfg)

print("System prompt the pipeline gives the model:\n")
print(DEFAULT_SYSTEM_PROMPT)

### Run the loop on a single prompt

Watch each turn: what the model wrote, and what the tests said. On a repair turn the model gets
the original task **plus the test failure** and tries again.

In [ ]:
def get_trace(idx):
    # Live or replay, this really runs Mellea's loop and the unit tests; only the model differs.
    trace = H.ivr_trace(problems[idx], idx, session_factory, cfg)
    if LIVE and os.environ.get("LAB_RECORD") == "1":   # instructors: refresh the recordings
        CACHE.mkdir(parents=True, exist_ok=True)
        (CACHE / f"trace_{idx:02d}.json").write_text(json.dumps(trace))
    return trace

trace = get_trace(14)   # StatevectorEstimator: the model trips on the V2 result format twice, then fixes it
H.print_trace(trace)

### Run it on all 20 prompts

Every prompt is independent, so we run them in parallel. Each attempt still has to pass the real unit
tests on your machine, so this takes a minute or two.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=8) as pool:
    traces = list(pool.map(get_trace, range(len(problems))))

solved = sum(1 for t in traces if t["solved_at_turn"])
first_try = sum(1 for t in traces if t["solved_at_turn"] == 1)
print(f"solved {solved}/{len(traces)}  ·  on the first try: {first_try}  ·  needed repairs: {solved - first_try}  ·  never solved: {len(traces) - solved}")
H.plot_ivr_timeline(traces, cfg.loop_budget); plt.show()

**Reading the chart:** each row is a prompt, each square is an attempt. A row that is all green at
turn 1 was easy for the model. Red squares followed by green are where the repair loop earned its
keep, and those red squares are exactly what we turn into **correction** data next.

## 3 · From one loop to three kinds of training data

The same run gives us three datasets, each training a different skill:

| Modality | Built from | The model is asked to... | Skill |
|---|---|---|---|
| **Question–Answer** | the original SDG | solve the task from scratch | *write* correct Qiskit |
| **Correction** | the last failed attempt + its error | fix nearly-correct code | *debug* and *repair* |
| **Fill-in-the-blank** | the passing answer with lines deleted | restore the missing lines | *recall* the exact Qiskit code |

Let's build all three with the results form the IVR loop: the same ones that produced our
full training sets.

Note: New files will be created in the `/data` folder.

In [ ]:
from dataclasses import asdict
from data_augmentation.models import CorrectionEntry
from data_augmentation.core.text import strip_assertion_internals
from data_augmentation.stages.correction import format_correction_prompt
from data_augmentation.stages import gold, fill_in_blank

OUT = Path("data/generated"); OUT.mkdir(parents=True, exist_ok=True)

# The "hub": one row per attempt, the format the correction stage writes.
hub = []
for t in traces:
    p = problems[t["problem_idx"]]
    solved_at = t["solved_at_turn"]
    passing = next((a["code"] for a in t["attempts"] if a["passed"]), "")
    for a in t["attempts"]:
        if not a["passed"] and a["code"]:
            hub.append(asdict(CorrectionEntry(
                prompt=format_correction_prompt(a["code"], p["prompt"], a["error"], a["deprecated_warnings"]),
                imports=p["imports"], unittest=p["unittest"], template="correction/" + p["template"],
                source_problem_idx=t["problem_idx"], attempt_number=a["turn"], error_type=a["error_type"],
                error_message=strip_assertion_internals(a["error"]), has_deprecated_apis=a["deprecated"],
                deprecated_api_warnings=a["deprecated_warnings"], solved=bool(solved_at), original_prompt=p["prompt"],
                broken_code=a["code"], correct_code=passing, solved_at_turn=solved_at,
                turns_to_fix=max(0, solved_at - a["turn"]))))
    if solved_at:
        hub.append(asdict(CorrectionEntry(
            prompt=p["prompt"], imports=p["imports"], unittest=p["unittest"], template=p["template"],
            source_problem_idx=t["problem_idx"], attempt_number=0, error_type="none", error_message="",
            has_deprecated_apis=False, deprecated_api_warnings="", solved=True, original_prompt=p["prompt"],
            broken_code="", correct_code=passing, solved_at_turn=solved_at, turns_to_fix=0)))
H.write_jsonl(hub, OUT / "correction_hub.jsonl")

# 1) QA: the original prompt + the verified answer
qa = [r for r in hub if r["attempt_number"] == 0]
H.write_jsonl(qa, OUT / "qa.jsonl")
# 2) Correction ("gold"): failed attempts from prompts that were eventually solved
n_gold = gold.extract_gold(str(OUT / "correction_hub.jsonl"), str(OUT / "correction.jsonl"))
# 3) Fill-in-the-blank: delete key lines from verified answers; keep only deletions that break the tests
n_fib = fill_in_blank.run_pipeline(str(OUT / "correction_hub.jsonl"), str(OUT / "fill_in_blank.jsonl"),
                                   types="import,call,combined", max_per_type=2, validate=True, max_workers=8)

corr = H.load_jsonl(OUT / "correction.jsonl")
fib = H.load_jsonl(OUT / "fill_in_blank.jsonl")
print(f"QA rows: {len(qa)}   correction rows: {len(corr)}   fill-in-the-blank rows: {len(fib)}")

In [ ]:
# Visualize the dataset generated using IVR loop
from collections import Counter
H.plot_dataset_funnel(len(problems), sum(1 for t in traces if t["solved_at_turn"]), len(qa), len(corr),
                      dict(Counter(r["deletion_type"] for r in fib))); plt.show()
H.plot_error_types(hub); plt.show()

### 3a · Question–Answer: *write it from scratch*

The prompt comes from the original data and we verify it using its unittest. This is the core dataset.

In [ ]:
if qa:
    r = random.choice(qa)
    print("PROMPT:\n" + r["prompt"])
    H.show_code(r["correct_code"], "Verified answer")

### 3b · Correction: *here's code that's almost right, fix it*

We take an attempt that **failed** inside the loop, from a prompt the model **eventually solved**, so
we know the bug is fixable. The new prompt contains the broken code and the error, but the unittest's
expected values are removed from the error message, so the model can't just copy the answer.

In [ ]:
if corr:
    r = random.choice(corr)
    print(f"error type: {r['error_type']}   ·   fixed {r['turns_to_fix']} turn(s) later   ·   deprecated APIs: {r['has_deprecated_apis']}")
    H.show_text(r["prompt"], "Correction prompt the model will be trained on", max_chars=3500)
    H.show_diff(r["broken_code"], r["correct_code"], "What changed between the failed and the passing attempt")
else:
    print("The model solved every prompt on its first try, so there were no failures to learn from.")

### 3c · Fill-in-the-blank: *restore the missing line*

We take verified code and **delete one key line**: an import, a key call (`AerSimulator(...)`,
`generate_preset_pass_manager(...)`, `SamplerV2(...)`...), or an import together with the line that uses it.
A deletion is kept only if the tests then **fail**; otherwise the line wasn't important.

In [ ]:
if fib:
    by_type = {}
    for r in fib:
        by_type.setdefault(r["deletion_type"], r)
    for kind, r in by_type.items():
        print(f"── deletion type: {kind}  ·  removed line(s):  {r['deleted_line']!r}")
    r = by_type.get("combined") or next(iter(by_type.values()))
    H.show_text(r["prompt"], f"Fill-in-the-blank prompt ({r['deletion_type']})", max_chars=3000)

### Compare the three modalities

The same task can be asked three ways, giving the model very different amounts of help.

In [ ]:
H.plot_modality_profile(qa, corr, fib); plt.show()

shared = sorted({r["source_problem_idx"] for r in qa} & {r["source_problem_idx"] for r in corr} & {r["source_problem_idx"] for r in fib})
if shared:
    k = shared[0]
    print(f"Prompt #{k} in all three forms:\n")
    for name, rows in [("QA", qa), ("Correction", corr), ("Fill-in-the-blank", fib)]:
        r = next(x for x in rows if x["source_problem_idx"] == k)
        print(f"=== {name} ===\n{r['prompt'][:700]}{' ...' if len(r['prompt']) > 700 else ''}\n")

## 4 · Using the data to train: reinforcement learning

With unit tests attached to every prompt, we don't need a learned reward model: **the tests
are the reward.** We used **GRPO** (Group Relative Policy Optimization):

1. For each prompt, the model generates a *group* of answers (we used 16).
2. Each answer is run against the unit tests: **1 if it passes, 0 if not**.
3. Answers that beat the group's average are reinforced; the rest are pushed down.

We trained Gemma-4-31B this way on **71,180** Qiskit prompts (QA, correction and fill-in-the-blank) for one epoch: 2,224 steps of
32 prompts × 16 answers. The chart shows its pass rate on **500 held-out problems from 10 templates it
never trained on**.

In [ ]:
res = json.loads(Path("data/training_results.json").read_text())
curve = res["validation_curve"]
H.plot_training_curve(curve["step"], curve["pass_rate"], base=res["four_sample_eval"]["base"]["mean_at_4"]); plt.show()

### Let's see the reward signal on a tiny scale

Here's what one GRPO step looks like for a single prompt: several answers, each scored by the tests,
and each answer's *advantage* (its reward minus the group average). Positive advantage → reinforced.

In [ ]:
def group_rollout(idx, n=6):
    p = problems[idx]
    cached = CACHE / f"group_{idx:02d}.json"
    if not LIVE:   # replay: the recorded answers, graded again right here
        codes = [g["code"] for g in json.loads(cached.read_text())]
    else:
        task = p["prompt"] + (f"\n\nThe following imports are available in the test environment:\n```python\n{p['imports'].strip()}\n```" if p["imports"].strip() else "")
        codes = [H.extract_code(H.ask(BASE_URL, BASE_MODEL, task, system=DEFAULT_SYSTEM_PROMPT, temperature=1.0)) for _ in range(n)]
    return [{"code": c, "reward": float(H.run_tests(c, p)[0])} for c in codes]   # the reward IS the unit tests

group = group_rollout(2)   # a prompt the base model gets right only some of the time
rewards = [g["reward"] for g in group]
mean = sum(rewards) / len(rewards)
for k, g in enumerate(group):
    print(f"answer {k}: reward {g['reward']:.0f}   advantage {g['reward'] - mean:+.2f}   {'▲ reinforce' if g['reward'] > mean else ('▼ discourage' if g['reward'] < mean else '— no signal')}")
print(f"\ngroup mean reward = {mean:.2f}.  If every answer passes (or every one fails), all advantages are 0 and the prompt teaches nothing.")

## 5 · Validation: did the model actually get better?

Training reward going up only shows the model got better at the training prompts. The real
question is whether it does better on problems it has **never seen**. First the large held-out
evaluation (500 problems, 4 tries each), then a head-to-head on the data **you** just generated.

In [ ]:
ev = res["four_sample_eval"]
table = pd.DataFrame({
    "model": ["base Gemma-4-31B", "tuned, step 625", "tuned, step 750", "tuned, step 825"],
    "pass rate (avg of 4 tries)": [ev[k]["mean_at_4"] for k in ["base", "step625", "step750", "step825"]],
    "pass@4 (≥1 of 4 correct)": [ev[k]["pass_at_4"] for k in ["base", "step625", "step750", "step825"]],
    "never solved (of 500)": [ev[k]["never_solved"] for k in ["base", "step625", "step750", "step825"]],
})
table.style.format({"pass rate (avg of 4 tries)": "{:.1%}", "pass@4 (≥1 of 4 correct)": "{:.1%}"}).hide(axis="index")

### Head-to-head on your generated data

We give **both models** the same prompts (one QA, correction and fill-in-the-blank set) with a single
try each, and run the unit tests.

**Be a careful scientist here:** the sample prompts were drawn from the same pool as the training data,
and 13 of the 20 QA prompts appear verbatim in the tuned model's RL training set. A gain on *those*
could just be memory. The cell after the chart splits the QA result into prompts the model trained on
and prompts it never saw, which is the comparison that actually tells you something.

In [ ]:
RECORDED_ANSWERS = json.loads((CACHE / "answers.json").read_text())   # model -> {prompt key: answer}

def score(model_url, model_name, rows, tag):
    label = tag.split("_")[0]
    def one(r):
        prob = {"imports": r["imports"], "unittest": r["unittest"]}
        if LIVE:
            ans = H.extract_code(H.ask(model_url, model_name, r["prompt"], system=DEFAULT_SYSTEM_PROMPT, temperature=0.0))
        else:   # replay: the model's recorded answer to this exact prompt (if it saw one)
            ans = RECORDED_ANSWERS[label].get(H.answer_key(r["prompt"]))
            if ans is None:
                return None
        return bool(H.run_tests(ans, prob)[0])   # graded here either way
    with ThreadPoolExecutor(max_workers=8) as pool:
        out = [x for x in pool.map(one, rows) if x is not None]
    return out

eval_sets = {"QA": [{**p, "prompt": p["prompt"]} for p in problems], "Correction": corr, "Fill-in-the-blank": fib}
results = {}
for label, url, name in [("base", BASE_URL, BASE_MODEL), ("tuned", TUNED_URL, TUNED_MODEL)]:
    results[label] = {m: score(url, name, rows, f"{label}_{m.replace(' ', '_').lower()}") for m, rows in eval_sets.items() if rows}
H.plot_base_vs_tuned(results); plt.show()

In [ ]:
seen = set(json.loads(Path("data/train_overlap.json").read_text())["seen_in_rl_training"])
split = pd.DataFrame([
    {"QA prompts": name, "n": len(idx),
     "base passed": sum(results["base"]["QA"][i] for i in idx),
     "tuned passed": sum(results["tuned"]["QA"][i] for i in idx)}
    for name, idx in [("in the RL training set", sorted(seen)),
                      ("never seen in training", [i for i in range(len(problems)) if i not in seen])]
])
split

**A note on the numbers:** with 20 prompts, a difference of one or two answers moves the bar by
5–10%, and the "never seen" group is only 7 prompts. Treat this head-to-head as intuition. The
reliable measurement is the 500-problem evaluation above: those problems come from **10 templates
that were held out of training entirely**, and the tuned model roughly doubles the base model there.

### 🔬 Held-out problems: templates the tuned model never trained on

These problems come from templates that were **held out of RL training entirely**. Each model got 4 tries
per problem; the cell below **re-runs every recorded answer against the unit tests on your machine**.

In [ ]:
show = json.loads((CACHE / "heldout_showcase.json").read_text())
rows_ = []
for it in show["items"]:
    prob = {"imports": it["imports"], "unittest": it["unittest"]}
    b = sum(H.run_tests(a["code"], prob)[0] for a in it["base"])
    t = sum(H.run_tests(a["code"], prob)[0] for a in it["tuned"])
    rows_.append({"modality": it["modality"], "template": it["template"], "base (of 4)": b, "tuned (of 4)": t})
heldout = pd.DataFrame(rows_)
print("total passing answers:  base", heldout["base (of 4)"].sum(), "  tuned", heldout["tuned (of 4)"].sum(),
      f"  (out of {4 * len(heldout)})")
heldout.style.background_gradient(subset=["base (of 4)", "tuned (of 4)"], cmap="Greens", vmin=0, vmax=4).hide(axis="index")

In [ ]:
# Look at one problem in detail: pick a row number from the table above.
k = 9
it = show["items"][k]
print(f"{it['modality']} · {it['template']}\n"); print(it["prompt"][:1500])
for who in ["base", "tuned"]:
    a = next((x for x in it[who] if x["passed"]), it[who][0])
    H.show_code(a["code"], f"{who.upper()} model: {'passed ✅' if a['passed'] else 'failed ❌'}")

## Wrap-up

| Stage | What it contributed |
|---|---|
| **Synthetic prompts + unit tests** | Unlimited, *verifiable* Qiskit tasks, so correctness is measured, not guessed |
| **Mellea IVR** | A structured loop that finds correct answers **and** records the near-misses |
| **QA · Correction · Fill-in-the-blank** | Three views of the same knowledge: *write*, *repair*, *recall* |
| **GRPO** | Tests as the reward, so the model is optimized directly for code that runs |
| **Validation** | Held-out problems confirm the gain is real: about **31% → 59%** pass rate, roughly double |

### Go further
* Check out our Mistral Qiskit model trained using a similar strategy! - https://huggingface.co/Qiskit/mistral-small-3.2-24b-qiskit
* Mellea: <https://github.com/generative-computing/mellea> · docs <https://docs.mellea.ai/>
* Qiskit: <https://quantum.cloud.ibm.com/docs/en/guides>
* Qiskit MCP servers: <https://github.com/Qiskit/mcp-servers>